# Problem Situation · Kickoff demo: from a map to an optimal tour
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Shown in class at the problem situation kickoff · about 20 min.*

This notebook walks through the whole pipeline of the problem situation on a **toy example in Paris**:

| Step | What we do | Tool |
|------|------------|------|
| 1 | Random delivery points on a map | `folium` |
| 2 | Driving **time and distance matrices** in one request | OSRM Table service |
| 3 | Export the data to **Excel** | `pandas` |
| 4 | Optimal **TSP tour** (asymmetric, MTZ) | `PuLP` + CBC |
| 5 | The tour on a map, following the real streets | OSRM Route service |
| 6 | **Traffic**: uniform factor vs congestion zone | your assumptions (TomTom data in the problem situation) |
| 7 | A first **CO₂** estimate | emission factors |

> **For your team:** replace the Paris points with your Monterrey depot and stops (Step 1), then reuse every
> other cell. Whether you need a TSP or a VRP is **your** modeling decision: this demo only shows the TSP.

Run the cells from top to bottom (Shift+Enter).

In [ ]:
# @title Setup (installs the packages on Colab; ~20 s the first time)
import importlib, subprocess, sys
import importlib.metadata
for pkg in ["folium", "openpyxl"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
try:                           # PuLP 4 + its CBC solver (pulp[cbc]); replaces an older PuLP 3
    _old = int(importlib.metadata.version("pulp").split(".")[0]) < 4 or importlib.util.find_spec("cbcbox") is None
except importlib.metadata.PackageNotFoundError:
    _old = True
if _old:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[cbc]>=4"], check=True)

import math, time
from pathlib import Path
import numpy as np, pandas as pd, requests, folium, pulp

OSRM = "https://router.project-osrm.org"   # free public demo server: few, small requests only
# Keyless basemap (OSM's volunteer tile server blocks notebooks; CARTO needs an API key)
MAP_TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"
MAP_ATTR = "Tiles &copy; Esri &mdash; Source: Esri, HERE, Garmin, OpenStreetMap contributors"
print("Setup done ✔")

---
## 1 · Delivery points
A depot (red) and 12 random stops in central Paris. The random generator has a **seed**, so everyone gets the same
points. OSRM will *snap* each point to the nearest street. The maps use the Esri World Street Map basemap, which
needs no API key (the OpenStreetMap volunteer tile server blocks notebook use).

In [ ]:
rng = np.random.default_rng(seed=2040)
N_STOPS = 12
depot = dict(name="Depot", lat=48.8443, lon=2.3744)          # near Gare de Lyon
stops = pd.DataFrame({
    "name": [f"Stop {i}" for i in range(1, N_STOPS + 1)],
    "lat": rng.uniform(48.835, 48.885, N_STOPS).round(5),
    "lon": rng.uniform(2.290, 2.400, N_STOPS).round(5),
})
locs = pd.concat([pd.DataFrame([depot]), stops], ignore_index=True)   # node 0 = depot
locs.index.name = "id"
display(locs)

m = folium.Map(location=[locs.lat.mean(), locs.lon.mean()], zoom_start=13, tiles=MAP_TILES, attr=MAP_ATTR)
for i, r in locs.iterrows():
    folium.Marker([r.lat, r.lon], tooltip=f"{i}: {r['name']}",
                  icon=folium.Icon(color="red" if i == 0 else "blue")).add_to(m)
m

**In the problem situation:** your points are real places (stores, schools, clinics…) chosen and justified by your team,
saved in `stops.csv` with their coordinates and operational data.

---
## 2 · Time and distance matrices with OSRM (one request)
The **Table service** returns the driving time (s) and distance (m) between **all pairs** of points.
Coordinates go as `lon,lat` separated by `;`. We call it **once** and save the result: later runs read the CSV files.
If the free public server is busy (a whole class calling it at once), the notebook uses a copy of the same
matrices stored in the course repository; set `SOURCE = "repo"` to use it directly.

In [ ]:
def osrm_table(locs):
    """One request to the OSRM Table service -> (time in minutes, distance in km) DataFrames."""
    coords = ";".join(f"{lon},{lat}" for lat, lon in zip(locs.lat, locs.lon))
    r = requests.get(f"{OSRM}/table/v1/driving/{coords}",
                     params={"annotations": "duration,distance"}, timeout=60)
    r.raise_for_status()
    data = r.json()
    assert data["code"] == "Ok", data
    T = pd.DataFrame(data["durations"], index=locs.index, columns=locs.index) / 60
    D = pd.DataFrame(data["distances"], index=locs.index, columns=locs.index) / 1000
    snap_m = pd.Series([s["distance"] for s in data["sources"]], index=locs.index)  # point -> nearest street
    return T, D, snap_m

# Where the matrices come from:
#   "osrm": call OSRM (default; if the server does not answer, falls back to the class copy)
#   "repo": download the class copy from the course repository (same Paris points, no OSRM call)
SOURCE = "osrm"
REPO = ("https://raw.githubusercontent.com/VincentALBoyer/Course-Materials/main/"
        "Undergraduate%20Level/OperationResearch/Data/")

def read_matrix(path):
    M = pd.read_csv(path, index_col=0); M.columns = M.columns.astype(int)
    return M

snap_m = None
if Path("time_matrix_min.csv").exists():                     # 1) reuse the matrices saved before
    T, D = read_matrix("time_matrix_min.csv"), read_matrix("distance_matrix_km.csv")
    print("Matrices loaded from the local CSV files")
else:
    if SOURCE == "osrm":
        try:                                                  # 2) one request to OSRM
            T, D, snap_m = osrm_table(locs)
            print("Matrices downloaded from OSRM")
        except Exception as err:
            print(f"OSRM did not answer ({type(err).__name__}): using the class copy instead")
            SOURCE = "repo"
    if SOURCE == "repo":                                      # 3) class copy from the course repository
        T, D = read_matrix(REPO + "time_matrix_min.csv"), read_matrix(REPO + "distance_matrix_km.csv")
        assert len(T) == len(locs), "the class copy only matches the seeded Paris points of Step 1"
        print("Matrices downloaded from the course repository")
    T.to_csv("time_matrix_min.csv"); D.to_csv("distance_matrix_km.csv")
if snap_m is not None:
    print(f"OSRM moved each point to the nearest street: at most {snap_m.max():.0f} m (point {snap_m.idxmax()})")
T.round(1)

**Checks you must always do:** the diagonal is zero, the matrix is **asymmetric** (one-way streets), and network
distances are usually longer than straight-line (haversine) distances. A ratio **below 1** means OSRM snapped a point
to a street far from where you placed it: look at the snapping distance above and fix the coordinates if needed.

In [ ]:
def haversine_km(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a.lat, a.lon, b.lat, b.lon))
    h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(h))

print("diagonal is zero:", bool((np.diag(T.values) == 0).all()))
asym = (T - T.T).abs()
i, j = np.unravel_index(asym.values.argmax(), asym.shape)
print(f"largest asymmetry: {T.iat[i, j]:.1f} min from {i} to {j}, but {T.iat[j, i]:.1f} min back")
ratio = [D.iat[a, b] / haversine_km(locs.iloc[a], locs.iloc[b])
         for a in range(len(locs)) for b in range(len(locs)) if a != b]
print(f"network / straight-line distance: mean {np.mean(ratio):.2f} (min {min(ratio):.2f}, max {max(ratio):.2f})")

---
## 3 · Export to Excel
One file with three sheets, to share the data or check it by hand.

In [ ]:
with pd.ExcelWriter("routing_data.xlsx") as xl:
    locs.to_excel(xl, sheet_name="locations")
    T.round(2).to_excel(xl, sheet_name="time_min")
    D.round(3).to_excel(xl, sheet_name="distance_km")
print("routing_data.xlsx written")
# On Colab: from google.colab import files; files.download("routing_data.xlsx")

---
## 4 · The optimal tour: an asymmetric TSP (MILP)
$x_{ij} = 1$ if the vehicle drives directly from $i$ to $j$; $c_{ij}$ = travel time (asymmetric).
$$\min \sum_{i \ne j} c_{ij} x_{ij} \quad \text{s.t.} \quad \sum_{j \ne i} x_{ij} = 1,\ \ \sum_{j \ne i} x_{ji} = 1
\ \ \forall i, \qquad u_i - u_j + n\,x_{ij} \le n - 1 \ \ \forall i \ne j,\ i, j \ne 0 \ \text{(MTZ)}$$
with $1 \le u_i \le n-1$ the position of stop $i$ in the tour.

In [ ]:
def solve_tsp(C, time_limit=60):
    """Asymmetric TSP with MTZ constraints. C: square DataFrame of costs. Returns (tour, status)."""
    nodes = list(C.index); n = len(nodes)
    prob = pulp.LpProblem("TSP", pulp.LpMinimize)
    x = {(i, j): prob.add_variable(f"x_{i}_{j}", cat="Binary") for i in nodes for j in nodes if i != j}
    u = {i: prob.add_variable(f"u_{i}", lowBound=1, upBound=n - 1) for i in nodes[1:]}
    prob += pulp.lpSum(C.at[i, j] * x[i, j] for (i, j) in x)
    for i in nodes:
        prob += pulp.lpSum(x[i, j] for j in nodes if j != i) == 1      # leave i once
        prob += pulp.lpSum(x[j, i] for j in nodes if j != i) == 1      # enter i once
    for i in nodes[1:]:
        for j in nodes[1:]:
            if i != j:
                prob += u[i] - u[j] + n * x[i, j] <= n - 1             # no subtours
    result = prob.solve(pulp.COIN_CMD(msg=False, timeLimit=time_limit))
    succ = {i: j for (i, j) in x if x[i, j].value() > 0.5}
    tour, k = [nodes[0]], succ[nodes[0]]
    while k != nodes[0]:
        tour.append(k); k = succ[k]
    return tour + [nodes[0]], result.status_str

def route_length(tour, M):
    return sum(M.at[a, b] for a, b in zip(tour, tour[1:]))

def check_tour(tour, nodes):
    """Independent check: starts/ends at the depot and visits every node exactly once."""
    assert tour[0] == tour[-1] == nodes[0], "must start and end at the depot"
    assert sorted(tour[:-1]) == sorted(nodes), "every node exactly once"
    return True

t0 = time.time()
tour, status = solve_tsp(T)
check_tour(tour, list(T.index))
print(f"status: {status}   solved in {time.time() - t0:.1f} s")
print("tour:", " → ".join(map(str, tour)))
print(f"total time: {route_length(tour, T):.1f} min   total distance: {route_length(tour, D):.2f} km")

**Benchmark:** a simple *nearest neighbor* tour (always go to the closest unvisited stop). The gap shows what the
optimization model is worth.

In [ ]:
def nearest_neighbor(C):
    nodes = list(C.index); tour, left = [nodes[0]], set(nodes[1:])
    while left:
        nxt = min(left, key=lambda j: C.at[tour[-1], j]); tour.append(nxt); left.remove(nxt)
    return tour + [nodes[0]]

nn = nearest_neighbor(T)
print(f"nearest neighbor: {route_length(nn, T):.1f} min   optimal TSP: {route_length(tour, T):.1f} min   "
      f"saving: {100 * (1 - route_length(tour, T) / route_length(nn, T)):.1f}%")

---
## 5 · The tour on a map
The **Route service** returns the street geometry of the tour (one request with the stops in tour order).

In [ ]:
def osrm_route_geometry(locs, tour):
    coords = ";".join(f"{locs.at[k, 'lon']},{locs.at[k, 'lat']}" for k in tour)
    r = requests.get(f"{OSRM}/route/v1/driving/{coords}",
                     params={"overview": "full", "geometries": "geojson"}, timeout=60)
    r.raise_for_status()
    route = r.json()["routes"][0]
    return [(lat, lon) for lon, lat in route["geometry"]["coordinates"]], route["duration"] / 60

def tour_map(locs, tour, color="green"):
    try:
        line, minutes = osrm_route_geometry(locs, tour)
        tip = f"{minutes:.1f} min (OSRM, free flow)"
    except Exception:                                    # OSRM unavailable: straight segments instead
        line = [(locs.at[k, "lat"], locs.at[k, "lon"]) for k in tour]
        tip, color = "straight segments (street geometry unavailable)", "gray"
    m = folium.Map(location=[locs.lat.mean(), locs.lon.mean()], zoom_start=13, tiles=MAP_TILES, attr=MAP_ATTR)
    folium.PolyLine(line, weight=5, color=color, tooltip=tip).add_to(m)
    for pos, k in enumerate(tour[:-1]):
        r = locs.loc[k]
        folium.Marker([r.lat, r.lon], tooltip=f"#{pos}: {r['name']}",
                      icon=folium.DivIcon(html=f'<div style="font-weight:bold;color:white;background:'
                                               f'{"#c0392b" if k == 0 else "#1a63a9"};border-radius:50%;'
                                               f'width:22px;height:22px;text-align:center;line-height:22px">{pos}</div>')
                      ).add_to(m)
    return m

tour_map(locs, tour)

---
## 6 · Traffic
OSRM times are **free-flow** (no traffic). Two simple ways to include congestion:

* **(a) Uniform factor:** every time × 1.3 at rush hour.
* **(b) Congestion zone:** only arcs that cross a congested area (here, whose midpoint is within 1.2 km of
  Châtelet) are slower (× 1.6).

In the problem situation you justify your factors with data, e.g. the **TomTom Traffic Index** (congestion by hour for
Monterrey) or travel times from the **TomTom Routing API** on a few pairs.

In [ ]:
# (a) uniform factor: the objective is just multiplied by 1.3, so the optimal tour cannot change
tour_a, _ = solve_tsp(1.3 * T)
print("(a) uniform ×1.3 -> same tour:", tour_a == tour,
      f"| time {route_length(tour, T):.1f} -> {route_length(tour_a, 1.3 * T):.1f} min")

# (b) congestion zone: slower arcs whose midpoint is within RADIUS_KM of Châtelet
center, RADIUS_KM, SLOWDOWN = pd.Series(dict(lat=48.8590, lon=2.3470)), 1.2, 1.6
def midpoint(a, b):
    return pd.Series(dict(lat=(locs.at[a, "lat"] + locs.at[b, "lat"]) / 2, lon=(locs.at[a, "lon"] + locs.at[b, "lon"]) / 2))
factor = pd.DataFrame(1.0, index=T.index, columns=T.columns)
for a in T.index:
    for b in T.columns:
        if a != b and haversine_km(midpoint(a, b), center) <= RADIUS_KM:
            factor.at[a, b] = SLOWDOWN
T_rush = T * factor
print(f"(b) {int((factor > 1).values.sum())} of {len(T) * (len(T) - 1)} arcs are congested")

tour_b, _ = solve_tsp(T_rush)
print("    free-flow tour :", " → ".join(map(str, tour)), f"| in rush hour it takes {route_length(tour, T_rush):.1f} min")
print("    rush-hour tour :", " → ".join(map(str, tour_b)), f"| {route_length(tour_b, T_rush):.1f} min")
print("    the tour changes:", tour_b != tour)

**Take-away:** a uniform factor changes *how long* the route takes, not *which* route is best. Only factors that
differ between areas or hours *can* change the decision (here it does; try other `RADIUS_KM` or `SLOWDOWN`
values), which is why good traffic data matters.

---
## 7 · A first CO₂ estimate
CO₂ per day = distance × consumption × emission factor. The values below are **indicative**: your team must find,
cite and justify its own (and the electricity factor depends on the country's grid).

In [ ]:
km = route_length(tour, D)
vehicles = pd.DataFrame([
    ("Gasoline van", 11.0, "L", 2.3),          # L/100 km, kg CO2 per L
    ("Diesel light truck", 14.0, "L", 2.7),
    ("Electric van (Mexican grid)", 20.0, "kWh", 0.43),   # kWh/100 km, kg CO2 per kWh
], columns=["vehicle", "use_per_100km", "unit", "kgCO2_per_unit"])
vehicles["energy_per_day"] = km * vehicles.use_per_100km / 100
vehicles["kgCO2_per_day"] = vehicles.energy_per_day * vehicles.kgCO2_per_unit
print(f"Tour length: {km:.2f} km")
vehicles.round(2)

---
## For your team
1. Replace Step 1 with your `stops.csv` (Monterrey depot and stops). Call OSRM **once** and keep the CSV files.
2. Decide your model: is a single tour enough, or do capacity, time windows, shift length or several vehicles make
   it a **VRP**? Extend `solve_tsp` accordingly (new variables and constraints) and extend `check_tour`.
3. Add your cost and environmental indicators to every solution, and compare the scenarios of your strategic question.